In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pandas as pd
from sklearn.feature_selection import mutual_info_classif            
import operator
import matplotlib.pyplot as plt
import numpy as np  
import pickle
import os
from huggingface_hub import login
from llama_index import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    ServiceContext,
)
from llama_index.node_parser import SentenceSplitter
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.embeddings.langchain import LangchainEmbedding
from llama_index.llms import HuggingFaceLLM
from transformers import AutoModelForCausalLM, AutoTokenizer
import faiss
from huggingface_hub import login

login(token="")


In [ ]:
from llama_index.schema import Document
import re
from langchain.embeddings import HuggingFaceEmbeddings


def clean_text(doc: Document) -> Document:
    # Remove headings like 2.2.3. or 1.1
    cleaned = re.sub(r"\b\d+(\.\d+)+\b", "", doc.text)
    return Document(text=cleaned, metadata=doc.metadata)

os.makedirs("pdfs", exist_ok=True)
documents_raw = SimpleDirectoryReader(input_dir="pdfs").load_data()

for doc in documents_raw:
    if not hasattr(doc, "metadata") or doc.metadata is None:
        doc.metadata = {}
    # Attach a name for later reference
    if hasattr(doc, "file_path"):
        doc.metadata["name"] = os.path.basename(doc.file_path)
    elif "file_path" in getattr(doc, "metadata", {}):
        doc.metadata["name"] = os.path.basename(doc.metadata["file_path"])
    else:
        doc.metadata["name"] = "Unknown document"

documents = [clean_text(doc) for doc in documents_raw]
print(f"Loaded and cleaned {len(documents)} documents.")

### 2. CHUNK DOCUMENTS, KEEP DOC NAME IN METADATA ###

parser = SentenceSplitter(chunk_size=256, chunk_overlap=20)
nodes = []
for doc in documents:
    doc_nodes = parser.get_nodes_from_documents([doc])
    for node in doc_nodes:
        node.metadata["doc_name"] = doc.metadata.get("name", "Unknown document")
    nodes.extend(doc_nodes)

### 3. CREATE VECTOR INDEX WITH EMBEDDINGS ###

embed_model = LangchainEmbedding(
    
    HuggingFaceEmbeddings(model_name="")


)
faiss_index = faiss.IndexFlatL2(384)
vector_store = FaissVectorStore(faiss_index=faiss_index)


In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1,3"
llm = HuggingFaceLLM(
    model_name="",
    tokenizer_name="",
    context_window=5800,
    #max_new_tokens=800,
    max_new_tokens=4000,
    generate_kwargs={"temperature": 0.0,
    "do_sample": False,
    #"top_p": 0.9,
    #"repetition_penalty": 1.2,
    #"eos_token_id": 2,
    #"pad_token_id":2,
    },
    device_map="auto",
    tokenizer_kwargs={"use_fast": True},
    model_kwargs={"torch_dtype": "auto"},
)
service_context = ServiceContext.from_defaults(
    embed_model=embed_model,
    llm=llm
)
index = VectorStoreIndex(nodes, vector_store=vector_store, service_context=service_context)



In [ ]:

doc_names = sorted({ n.metadata.get("doc_name", "Unknown document") for n in nodes })
print("Documents in memory:")
for name in doc_names:
    print("-", name)



In [ ]:
# ===================== DIAGNOSTIC: REVIEW → (Q pop-ups) → REPORT =====================
# - Prints every prompt/response/decision
# - Catches and prints full traceback + likely cause
# - Toggle pop-ups with USE_TK_POPUPS

import re, sys, traceback
from typing import List, Dict

USE_TK_POPUPS = True   # <- set to False to avoid Tk entirely and use text input

# -------- Preflight checks --------
print("=== PREFLIGHT ===")
print("Python exe:", sys.executable)
# Tk availability
try:
    import tkinter as _tk  # only to probe; actual import later
    print("Tkinter: AVAILABLE")
except Exception as e:
    print("Tkinter: NOT AVAILABLE ->", repr(e))
# LlamaIndex index presence
try:
    index  # must exist from your previous cell
    print("LlamaIndex `index`: OK")
except NameError:
    print("LlamaIndex `index`: MISSING. Build it in a prior cell before running this.")
    raise

print("\n=== START RUN ===")

# ---------- Pop-up helper (Tkinter). Falls back to input() if disabled or fails ----------
def ask_many(questions: List[str]) -> Dict[str, str]:
    answers: Dict[str, str] = {}
    if not questions:
        return answers

    if USE_TK_POPUPS:
        try:
            import tkinter as tk
            from tkinter import simpledialog, messagebox
            root = tk.Tk(); root.withdraw()
            for q in questions:
                ans = simpledialog.askstring("Clarification needed", q)  # blocking dialog
                if ans is None or not str(ans).strip():
                    ans = "Not available"
                answers[q] = ans.strip()
            try:
                messagebox.showinfo("Info", "Thanks! Generating the final report…")
            except Exception:
                pass
            root.destroy()
            return answers
        except Exception as e:
            print("\n[Pop-up error] Falling back to inline input:", repr(e))

    # Fallback: inline text input (VS Code shows an input bar)
    for q in questions:
        ans = input(f"{q}\n> ").strip() or "Not available"
        answers[q] = ans
    return answers

# ---------- Tiny case container ----------
class CaseState:
    def __init__(self, case_id: str):
        self.case_id = case_id
        self.wscores_text = ""
        self.answers: Dict[str, str] = {}
        self.history: List[str] = []
        self.sources_whitelist: List[str] = []

    def as_context(self) -> str:
        lines = [f"[CASE_ID] {self.case_id}"]
        if self.wscores_text:
            lines.append("[W-SCORES]")
            lines.append(self.wscores_text)
        if self.answers:
            lines.append("[CLINICAL/EXTRA DATA]")
            for k, v in self.answers.items():
                lines.append(f"- {k}: {v}")
        if self.sources_whitelist:
            lines.append("[YOU MAY CITE ONLY THESE DOCUMENTS]")
            for i, nm in enumerate(self.sources_whitelist, 1):
                lines.append(f"{i}. {nm}")
        return "\n".join(lines)

# ---------- LLM caller ----------
def llm_call_with_context(query_engine, system_prompt: str, user_prompt: str, history: List[str]) -> str:
    MAX_HISTORY_TURNS = 6
    convo = "\n\n".join(history[-MAX_HISTORY_TURNS:])
    full_prompt = f"{system_prompt}\n\n{convo}\n\n{user_prompt}" if convo else f"{system_prompt}\n\n{user_prompt}"
    return str(query_engine.query(full_prompt))

# ---------- Prompts (prints decision & questions explicitly) ----------
REVIEW_SYSTEM = """You are a careful radiology assistant.
Use ONLY the provided RAG sources if you cite. If a claim is not in the sources, say: [Not supported in available sources].
Be concise. Ask ONLY clinically impactful questions relevant to the NIA-AA framework and MRI dementia workup.
This stage MUST NOT produce findings or an impression.
Always include 1–4 clarifying questions (skip items already present in [CLINICAL/EXTRA DATA]) and end with: END: WAITING_FOR_ANSWERS or END: READY_TO_PROCEED.
"""

REVIEW_USER_TMPL = """[TASK]
1) Review the W-scores below.
2) Identify regions with ≥2 abnormal values (|W|>2). Summarize briefly under [SUMMARY].
3) Decide readiness under [DECISION] (see format).
4) Ask 1-4 focused clarifying questions related to only MRI imaging biomarker to provide better interpretation (NIA-AA / MRI-guideline perspective) under [CLARIFY].
5) Do NOT ask about information already supplied under [CLINICAL/EXTRA DATA].
6) Do NOT generate findings or an impression.

[DECISION FORMAT]
- Qualifying_regions_count: <integer>
- Proceed: YES or NO

[OUTPUT FORMAT]
[SUMMARY]
- 1–3 bullets.

[DECISION]
- Qualifying_regions_count: <integer>
- Proceed: YES|NO

[CLARIFY]
- One question per line (max 4).

END: WAITING_FOR_ANSWERS or END: READY_TO_PROCEED

{case_context}
"""

REPORT_SYSTEM = """You are an expert radiology assistant.
Cite ONLY the allowed documents (file names provided). If a claim is not supported in those sources, write: [Not supported in available sources].
You MUST explicitly integrate the items under [CLINICAL/EXTRA DATA] into your synthesis.
If you do not use any [CLINICAL/EXTRA DATA], you MUST write: [WARNING: No clinical data used].
Do not make a definitive diagnosis. Avoid boilerplate unless explicitly supported by sources.
Generate a clear trainee-oriented report (FINDINGS + IMPRESSION)."""

REPORT_USER_TMPL = """[TASK]
Use the W-scores AND the items listed under [CLINICAL/EXTRA DATA] to produce the report based only on MRI Imaging.

[DATA_USED]
- List 3–8 bullets, each: "Clinical datum → how it changes interpretation (weighting/likelihood/limitations)".
- Only include items that appear under [CLINICAL/EXTRA DATA].
- If none are used, write exactly: [WARNING: No clinical data used]

TASK:
1. Write **FINDINGS**: summarize abnormalities with W-scores. For each region,specify the pattern, severity, and w-scores, and briefly discuss the potential clinical and pathophysiological relevance, citing one of the six sources by filename.
2. Write **IMPRESSION**: 
    - Provide a **multi-paragraph, guideline-based summary**:
    - Integrate findings with the Alzheimer’s Dementia papers as diagnostic framework.
    - Discuss the pattern and severity of atrophy and its implications for neurodegenerative diseases but do not diagnose any disease.
    - Add a short paragraph explaining the clinical significance about each regions in the findings in detail based on the papers
3. Every claim must be followed by a citation in the format: (Filename, page/section if available).  

Do NOT cite anything outside the six listed documents. Do NOT invent facts.
Ask more information if you require to do impression


OUTPUT FORMAT:
FINDINGS:
- …

IMPRESSION:
- …
[IMPORTANT]Show step by step before giving the output.

{case_context}
"""

# ---------- Parsers ----------
def parse_decision(text: str):
    m_cnt = re.search(r"Qualifying_regions_count\s*:\s*(\d+)", text, flags=re.I)
    m_proceed = re.search(r"Proceed\s*:\s*(YES|NO)", text, flags=re.I)
    return {
        "qual_count": int(m_cnt.group(1)) if m_cnt else None,
        "proceed": m_proceed.group(1).upper() if m_proceed else None,
        "ready_token": "READY_TO_PROCEED" in text.upper(),
        "waiting_token": "WAITING_FOR_ANSWERS" in text.upper(),
    }

def extract_questions(text: str) -> List[str]:
    match = re.search(r"\[clarify\](.*?)(?:\n\s*end\s*:\s*(?:waiting_for_answers|ready_to_proceed)|\Z)", text, flags=re.I | re.S)
    if not match:
        return []
    block = match.group(1)
    qs: List[str] = []
    for line in block.splitlines():
        s = line.strip()
        if not s:
            continue
        s = s.lstrip("-•*0123456789). ").strip()
        if s:
            qs.append(s)
    # de-duplicate while preserving order
    seen, out = set(), []
    for q in qs:
        k = q.lower()
        if k not in seen:
            seen.add(k); out.append(q)
    return out[:4]

# ---------- Configure your case ----------
case = CaseState(case_id="PT-001")
case.wscores_text = """\
mild pathology in atrophied Left Temporal Lobe: (volume w-score: -2.97, relevance w-score: -3.66, Cortical Thickness W-scores: -2.02)
Strong pathology in atrophied Right Temporal Lobe: (volume w-score: -4.57, Cortical Thickness W-scores: -4.70)
mild pathology in atrophied Left Hippocampus: (volume w-score: -2.51, relevance w-score: -2.70)
Moderate pathology in atrophied Left Entorhinal: (volume w-score: -3.06, Cortical Thickness W-scores: -3.03)
"""
case.sources_whitelist = [
   "Alzheimer’s Dementia - 2011 - Jack - Introduction to the recommendations from the NIA-AA.pdf",
   "Alzheimer’s Dementia - 2011 - McKhann - The diagnosis of dementia due to AD.pdf",
   "Alzheimer’s Dementia - 2011 - Albert - The diagnosis of mild cognitive impairment due to AD.pdf",
   "Alzheimer’s Dementia - 2011 - Sperling - Toward defining the preclinical stages of AD.pdf",
   "DGN Guidelines Diagnosis.pdf",
   "MRI for diagnosing dementia - update.pdf",
]

# ---------- Run (with robust error reporting) ----------
try:
    # REVIEW
    qe = index.as_query_engine(response_mode="compact")
    review_user = REVIEW_USER_TMPL.format(case_context=case.as_context())
    print("\n===== REVIEW PROMPT (USER) =====\n", review_user)
    print("\n===== REVIEW SYSTEM =====\n", REVIEW_SYSTEM)

    review_out = llm_call_with_context(qe, REVIEW_SYSTEM, review_user, case.history)
    case.history += [f"USER:\n{review_user}", f"ASSISTANT:\n{review_out}"]

    print("\n===== REVIEW OUTPUT (RAW) =====\n", review_out)

    decision = parse_decision(review_out)
    questions = extract_questions(review_out)

    print("\n===== PARSED DECISION =====")
    print("Qualifying_regions_count:", decision["qual_count"])
    print("Proceed (model):", decision["proceed"])
    print("End token READY_TO_PROCEED:", decision["ready_token"])
    print("End token WAITING_FOR_ANSWERS:", decision["waiting_token"])

    print("\n===== CLARIFYING QUESTIONS (from model) =====")
    if questions:
        for i, q in enumerate(questions, 1):
            print(f"{i}) {q}")
    else:
        print("(None provided by the model)")

    # Ask (pop-ups or inline)
    if questions:
        answers = ask_many(questions)
        print("\n===== YOUR ANSWERS =====")
        for i, q in enumerate(questions, 1):
            print(f"{i}) {q}\n   -> {answers.get(q, 'Not available')}")
        case.answers.update(answers)
        qa_block = "USER:\n[CLARIFICATION ANSWERS]\n" + "\n".join(
            f"- {q}: {answers[q]}" for q in questions
        )
        case.history.append(qa_block)
    else:
        print("\n(Proceeding without clarifications)")

    # REPORT
    qe2 = index.as_query_engine(response_mode="compact")
    report_user = REPORT_USER_TMPL.format(case_context=case.as_context())
    print("\n===== REPORT PROMPT (USER) =====\n", report_user)
    print("\n===== REPORT SYSTEM =====\n", REPORT_SYSTEM)

    final_report = llm_call_with_context(qe2, REPORT_SYSTEM, report_user, case.history)
    case.history += [f"USER:\n{report_user}", f"ASSISTANT:\n{final_report}"]

    print("\n===== FINAL REPORT =====\n", final_report)

    # Quick integration check
    used = "CONFIRM_USED_CLINICAL_DATA: YES" in final_report
    warn = "[WARNING: No clinical data used]" in final_report
    print("\n===== INTEGRATION CHECK =====")
    if used and not warn:
        print("✓ The model claims it used your clinical answers.")
    elif warn:
        print("⚠ The model says it did NOT use any clinical data. Consider revising answers or prompt.")
    else:
        print("❓ Could not verify usage from the report text. Check the [DATA_USED] section above.")

except Exception as e:
    print("\n===== ERROR CAUGHT =====")
    print(type(e).__name__ + ":", e)
    print("\n----- TRACEBACK -----")
    traceback.print_exc()
    print("\n----- LIKELY CAUSES & FIXES -----")
    msg = str(e).lower()

    if "nameerror" in str(type(e)).lower() and "index" in msg:
        print("* `index` is not defined in this kernel. Build your LlamaIndex in a prior cell.")
    elif "tkinter" in msg or "tclerror" in msg:
        print("* Tk pop-ups failed (e.g., no DISPLAY or Tk not installed). Set USE_TK_POPUPS=False to use inline input.")
        print("  - Conda: `conda install -c anaconda tk`")
        print("  - Ubuntu: `sudo apt-get install python3-tk`")
        print("  - WSL/SSH: use inline input (no GUI) or enable X forwarding.")
    elif "attributeerror" in str(type(e)).lower() and "query" in msg:
        print("* Your LlamaIndex version may return a query engine with a different interface.")
        print("  Try: qe = index.as_query_engine(); response = qe.query(full_prompt)")
    elif "runtimeerror" in str(type(e)).lower():
        print("* The model may not have produced expected sections. Check 'REVIEW OUTPUT (RAW)' above.")
    else:
        print("* Review the traceback above; the printouts show the exact prompts and raw model outputs.")

